# HOGENOM likelihood optimization

This notebook uses the lean `GeneReconModel` API to load the HOGENOM species tree and gene-tree distributions, evaluate the full negative log-likelihood, and optimize the likelihood end to end with Adam.

Set `MODE` to `global`, `specieswise`, or `genewise`.

In [ ]:
from pathlib import Path
import json
import math
import os
import subprocess
import time

import sys

import numpy as np
import torch

CWD = Path.cwd()
REPO = CWD if (CWD / "gpurec").exists() else CWD.parent if (CWD.parent / "gpurec").exists() else CWD
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from gpurec import GeneReconModel, SolverOptions, clamp_log_rate_, project_rate_gradient_

torch.set_float32_matmul_precision("high")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("The current lean HOGENOM likelihood path requires CUDA/Triton.")
DEVICE


## Paths and run settings

In [ ]:
def find_hogenom_root() -> Path:
    env_root = os.environ.get("HOGENOM_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([
        REPO / "tests/data",
        REPO / "tests/data/HOGENOM/hogenom",
        REPO / "tests/data/HOGENOM",
        REPO / "tests/data/hogenom",
        REPO / "data/HOGENOM/hogenom",
    ])
    for candidate in candidates:
        if (candidate / "hogenom_S.tree").exists() and (candidate / "hogenom_trees").is_dir():
            return candidate
    raise FileNotFoundError(
        "Set HOGENOM_ROOT, or set both HOGENOM_SPECIES_TREE and "
        "HOGENOM_GENE_TREE_DIR, before running this notebook."
    )

species_tree_env = os.environ.get("HOGENOM_SPECIES_TREE")
gene_tree_dir_env = os.environ.get("HOGENOM_GENE_TREE_DIR")
if species_tree_env or gene_tree_dir_env:
    if not (species_tree_env and gene_tree_dir_env):
        raise ValueError("Set both HOGENOM_SPECIES_TREE and HOGENOM_GENE_TREE_DIR, or neither.")
    SPECIES_TREE = Path(species_tree_env).expanduser()
    GENE_TREE_DIR = Path(gene_tree_dir_env).expanduser()
    HOGENOM_ROOT = GENE_TREE_DIR.parent
else:
    HOGENOM_ROOT = find_hogenom_root()
    SPECIES_TREE = HOGENOM_ROOT / "hogenom_S.tree"
    GENE_TREE_DIR = HOGENOM_ROOT / "hogenom_trees"

MODE = os.environ.get("GPUREC_MODE", "specieswise")  # global, specieswise, or genewise
MAX_FAMILIES = None  # set to an int for a fast smoke run

STEPS = 1000
LR = 2
PRINT_EVERY = 10
CLIP_GRAD_NORM = 1_000.0
MIN_RATE = 1e-10
MAX_RATE = 2.0
OPTIMIZER_NAME = "Adam-LBFGSB"  # Adam, Adagrad, LBFGSB, or chains like Adam-LBFGSB

# MAP penalty controls. The default lambda comes from the bounded CV pilot in
# benchmarks/large_dataset_capacity/reports/hogenom_penalty_cv_subagent.md.
PENALTY_KIND = os.environ.get("GPUREC_PENALTY_KIND", "l2")  # none or l2
L2_PENALTY_LAMBDA = float(os.environ.get("GPUREC_L2_PENALTY_LAMBDA", "0.01"))
L2_PENALTY_LAMBDAS = os.environ.get("GPUREC_L2_PENALTY_LAMBDAS", "0.01,0.01,0.01")  # optional D,T,L strengths
TREE_PENALTY_LAMBDAS = os.environ.get("GPUREC_TREE_PENALTY_LAMBDAS", "1,30,10")  # optional D,T,L unit-branch GBM strengths
ROOT_PENALTY_LAMBDAS = os.environ.get("GPUREC_ROOT_PENALTY_LAMBDAS", "1,30,10")  # optional D,T,L root-anchor strengths
L2_TARGET_RATE = float(os.environ.get("GPUREC_L2_TARGET_RATE", "0.05"))

# L-BFGS-B controls. The SciPy path optimizes bounded flattened theta values
# after any first-order warmup stages in the selected chain. SciPy's gtol is
# an absolute infinity norm of the projected gradient, so tiny values such as
# 1e-5 are usually too strict for this summed fp32/iterative likelihood.
LBFGSB_MAXLS = 50
LBFGSB_MAXCOR = 20
LBFGSB_GTOL = 1.0
LBFGSB_FTOL = 1e-6
LBFGSB_MAXFUN_MULTIPLIER = 25

# Adaptive solver controls. When the loss stalls, increase solver accuracy
# and reset optimizer state so stale moments do not dominate the new regime.
STALL_PATIENCE = 20
STALL_REL_IMPROVEMENT = 1e-3
STALL_ABS_IMPROVEMENT = 0.0
PI_ITERS_INCREMENT = 16
PI_ITERS_MAX = 32
NEUMANN_TERMS_INCREMENT = 16
NEUMANN_TERMS_MAX = 32
RESET_OPTIMIZER_ON_STALL = True

# Solver controls. These can also be changed during optimization with
# model.configure_solver(...) or direct model.solver_options.* assignment.
E_INIT = -1000
E_MAX_ITER = 2000
E_TOL = 1e-8
PI_ITERS = 16
NEUMANN_TERMS = 16
SELF_LOOP_SOLVER = os.environ.get("GPUREC_SELF_LOOP_SOLVER", "gmres")  # neumann or gmres
BICGSTAB_MAX_ITER = 500
BICGSTAB_TOL = 1e-7
BICGSTAB_BREAKDOWN_TOL = 1e-30
ADJOINT_PRUNING_THRESHOLD = 1e-6
USE_ADJOINT_PRUNING = True
PIBAR_SIDE_THRESHOLD = 0.0

species_tree = SPECIES_TREE
gene_trees = sorted(GENE_TREE_DIR.glob("*.trees"))
if MAX_FAMILIES is not None:
    gene_trees = gene_trees[:MAX_FAMILIES]

if not species_tree.exists():
    raise FileNotFoundError(f"species tree not found: {species_tree}")
if not gene_trees:
    raise FileNotFoundError(f"no *.trees files found in {GENE_TREE_DIR}")

{
    "mode": MODE,
    "device": DEVICE,
    "species_tree": str(species_tree),
    "gene_tree_dir": str(GENE_TREE_DIR),
    "families": len(gene_trees),
    "min_rate": MIN_RATE,
    "max_rate": MAX_RATE,
    "optimizer": OPTIMIZER_NAME,
    "penalty_kind": PENALTY_KIND,
    "l2_penalty_lambda": L2_PENALTY_LAMBDA,
    "l2_penalty_lambdas": L2_PENALTY_LAMBDAS,
    "tree_penalty_lambdas": TREE_PENALTY_LAMBDAS,
    "root_penalty_lambdas": ROOT_PENALTY_LAMBDAS,
    "l2_target_rate": L2_TARGET_RATE,
    "stall_patience": STALL_PATIENCE,
    "solver": {
        "e_max_iter": E_MAX_ITER,
        "e_tol": E_TOL,
        "pi_iters": PI_ITERS,
        "neumann_terms": NEUMANN_TERMS,
        "self_loop_solver": SELF_LOOP_SOLVER,
        "bicgstab_max_iter": BICGSTAB_MAX_ITER,
        "bicgstab_tol": BICGSTAB_TOL,
    },
}


## Native extension check

In [ ]:
from gpurec.core.scheduling import batching as _gpurec_batching

try:
    _gpurec_batching._load_native_module()
except ImportError:
    subprocess.run(["cargo", "build", "--release"], cwd=REPO / "crates/gpurec-preprocess", check=True)
    _gpurec_batching._load_native_module()

True

## Build the model

In [ ]:
torch.cuda.empty_cache()
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()
solver_options = SolverOptions(
    e_init=E_INIT,
    e_max_iter=E_MAX_ITER,
    e_tol=E_TOL,
    pi_iters=PI_ITERS,
    neumann_terms=NEUMANN_TERMS,
    self_loop_solver=SELF_LOOP_SOLVER,
    bicgstab_max_iter=BICGSTAB_MAX_ITER,
    bicgstab_tol=BICGSTAB_TOL,
    bicgstab_breakdown_tol=BICGSTAB_BREAKDOWN_TOL,
    adjoint_pruning_threshold=ADJOINT_PRUNING_THRESHOLD,
    use_adjoint_pruning=USE_ADJOINT_PRUNING,
    pibar_side_threshold=PIBAR_SIDE_THRESHOLD,
)
model = GeneReconModel(
    species_tree,
    gene_trees,
    mode=MODE,
    device=DEVICE,
    family_chunk_size=300,
    clade_budget=315_000,
    batch_packing="depth_first_fit",
    max_wave_size=8192,
    solver_options=solver_options,
)
clamp_log_rate_(model.theta, min_rate=MIN_RATE, max_rate=MAX_RATE)
if DEVICE == "cuda":
    torch.cuda.synchronize()
build_s = time.perf_counter() - t0

{
    "build_s": build_s,
    "theta_shape": tuple(model.theta.shape),
    "batches": [len(batch) for batch in model.family_batches],
    "waves": [len(static.wave_layout["wave_metas"]) for static in model.batch_statics],
    "solver_options": model.solver_options.__dict__,
}

## Initial likelihood and gradient

In [ ]:
model.zero_grad(set_to_none=True)
t0 = time.perf_counter()
loss0 = model()
loss0.backward()
if DEVICE == "cuda":
    torch.cuda.synchronize()
eval_s = time.perf_counter() - t0

grad_norm0 = float(model.theta.grad.detach().norm().cpu())
peak_gib = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE == "cuda" else None

{
    "initial_nll_bits": float(loss0.detach().cpu()),
    "grad_norm": grad_norm0,
    "eval_s": eval_s,
    "peak_gib": peak_gib,
}

## Optimize

Set `OPTIMIZER_NAME` to a single optimizer (`Adam`, `Adagrad`, `LBFGSB`) or a short chain such as `Adam-LBFGSB`. A first-order stage before another stage runs until the adaptive solver reaches `NEUMANN_TERMS_MAX`, then the next stage continues from the same `theta`.


In [ ]:
FIRST_ORDER_OPTIMIZERS = {
    "adam": torch.optim.Adam,
    "adagrad": torch.optim.Adagrad,
}

OPTIMIZER_ALIASES = {
    "adam": ("adam",),
    "adagrad": ("adagrad",),
    "lbfgsb": ("lbfgsb",),
    "lbfgs-b": ("lbfgsb",),
    "l-bfgs-b": ("lbfgsb",),
    "bfgsb": ("lbfgsb",),
    "adam-lbfgsb": ("adam", "lbfgsb"),
    "adam-lbfgs-b": ("adam", "lbfgsb"),
    "adam-l-bfgs-b": ("adam", "lbfgsb"),
    "adam-then-lbfgsb": ("adam", "lbfgsb"),
    "adam-then-l-bfgs-b": ("adam", "lbfgsb"),
    "adagrad-lbfgsb": ("adagrad", "lbfgsb"),
    "adagrad-lbfgs-b": ("adagrad", "lbfgsb"),
    "adagrad-l-bfgs-b": ("adagrad", "lbfgsb"),
    "adagrad-then-lbfgsb": ("adagrad", "lbfgsb"),
    "adagrad-then-l-bfgs-b": ("adagrad", "lbfgsb"),
}


def normalize_optimizer_stage(name: str) -> str:
    key = str(name).strip().lower().replace("_", "-").replace(" ", "")
    if key in OPTIMIZER_ALIASES and len(OPTIMIZER_ALIASES[key]) == 1:
        return OPTIMIZER_ALIASES[key][0]
    raise ValueError(f"unknown optimizer stage {name!r}")


def parse_optimizer_chain(name: str) -> tuple[str, ...]:
    key = str(name).strip().lower().replace("_", "-").replace(" ", "")
    if key in OPTIMIZER_ALIASES:
        return OPTIMIZER_ALIASES[key]
    key = key.replace("then", "->")
    for sep in ("->", ",", ";", "+"):
        if sep in key:
            stages = tuple(normalize_optimizer_stage(part) for part in key.split(sep) if part)
            if stages:
                return stages
    return (normalize_optimizer_stage(key),)


def stage_label(stage: str) -> str:
    return "LBFGSB" if stage == "lbfgsb" else stage.capitalize()


def chain_label(chain) -> str:
    return "-".join(stage_label(stage) for stage in chain)


def log2_bounds(min_rate, max_rate):
    lower = math.log2(float(min_rate))
    upper = None if max_rate is None else math.log2(float(max_rate))
    return lower, upper


def lbfgsb_bounds(theta: torch.Tensor, *, min_rate, max_rate):
    lower, upper = log2_bounds(min_rate, max_rate)
    return [(lower, upper)] * int(theta.numel())


@torch.no_grad()
def load_flat_theta_(model, flat_theta, *, min_rate, max_rate):
    flat_theta = np.asarray(flat_theta, dtype=np.float64)
    theta_cpu = torch.from_numpy(flat_theta.reshape(tuple(model.theta.shape)))
    model.theta.copy_(theta_cpu.to(device=model.theta.device, dtype=model.theta.dtype))
    clamp_log_rate_(model.theta, min_rate=min_rate, max_rate=max_rate)


def flat_theta_numpy(model) -> np.ndarray:
    return model.theta.detach().cpu().double().numpy().reshape(-1).copy()


def l2_penalty_lambdas(theta):
    raw = str(L2_PENALTY_LAMBDAS).strip()
    if not raw:
        return theta.new_full((theta.shape[-1],), float(L2_PENALTY_LAMBDA))
    values = [float(part.strip()) for part in raw.split(",") if part.strip()]
    if len(values) != theta.shape[-1]:
        raise ValueError(f"L2_PENALTY_LAMBDAS must have {theta.shape[-1]} comma-separated values")
    return torch.as_tensor(values, dtype=theta.dtype, device=theta.device)


def optional_event_lambdas(theta, raw):
    raw = str(raw).strip()
    if not raw:
        return None
    values = [float(part.strip()) for part in raw.split(",") if part.strip()]
    if len(values) != theta.shape[-1]:
        raise ValueError(f"event lambda list must have {theta.shape[-1]} comma-separated values")
    return torch.as_tensor(values, dtype=theta.dtype, device=theta.device)


def theta_penalty_bits(theta):
    kind = str(PENALTY_KIND).strip().lower()
    if kind in {"", "none", "off", "0"}:
        return theta.new_zeros(())
    if kind != "l2":
        raise ValueError(f"unsupported PENALTY_KIND={PENALTY_KIND!r}; supported: none, l2")
    target = theta.new_full(theta.shape, math.log2(float(L2_TARGET_RATE)))
    penalty = 0.5 * torch.sum(l2_penalty_lambdas(theta) * (theta - target) ** 2)
    tree_lambdas = optional_event_lambdas(theta, TREE_PENALTY_LAMBDAS)
    root_lambdas = optional_event_lambdas(theta, ROOT_PENALTY_LAMBDAS)
    parent = model.species_helpers["sp_parent"].to(device=theta.device, dtype=torch.long)
    if tree_lambdas is not None:
        edge_mask = parent >= 0
        penalty = penalty + 0.5 * torch.sum(tree_lambdas * (theta[edge_mask] - theta[parent[edge_mask]]) ** 2)
    if root_lambdas is not None:
        root_mask = parent < 0
        penalty = penalty + 0.5 * torch.sum(root_lambdas * (theta[root_mask] - target[root_mask]) ** 2)
    return penalty


def evaluate_loss_and_grad(model, *, min_rate, max_rate, check_finite=True):
    model.zero_grad(set_to_none=True)
    model.theta.grad = None
    raw_nll = model()
    penalty = theta_penalty_bits(model.theta)
    objective = raw_nll + penalty
    objective.backward()
    if model.theta.grad is None:
        raise RuntimeError("missing theta gradient")
    grad = model.theta.grad.detach()
    if check_finite and (not torch.isfinite(objective).all() or not torch.isfinite(grad).all()):
        raise FloatingPointError("non-finite loss or gradient")
    return objective, grad, raw_nll.detach(), penalty.detach()


def projected_grad_stats(theta, grad, *, min_rate, max_rate, bound_atol=1e-6):
    projected = grad.detach().clone()
    project_rate_gradient_(theta, projected, min_rate=min_rate, max_rate=max_rate)
    flat = projected.reshape(-1)

    lower, upper = log2_bounds(min_rate, max_rate)
    theta_detached = theta.detach()
    at_lower = theta_detached <= lower + bound_atol
    if upper is None:
        at_upper = torch.zeros_like(theta_detached, dtype=torch.bool)
    else:
        at_upper = theta_detached >= upper - bound_atol
    free = ~(at_lower | at_upper)

    return {
        "projected_grad_norm": float(flat.norm().detach().cpu()),
        "projected_grad_linf": float(flat.abs().max().detach().cpu()) if flat.numel() else 0.0,
        "active_params": int((at_lower | at_upper).sum().detach().cpu()),
        "active_lower_params": int(at_lower.sum().detach().cpu()),
        "active_upper_params": int(at_upper.sum().detach().cpu()),
        "free_params": int(free.sum().detach().cpu()),
        "param_count": int(theta_detached.numel()),
    }


def projected_grad_stats_for_diagnostics(model, *, min_rate, max_rate):
    if model.theta.grad is None:
        return {
            "projected_grad_norm": float("nan"),
            "projected_grad_linf": float("nan"),
            "active_params": 0,
            "active_lower_params": 0,
            "active_upper_params": 0,
            "free_params": int(model.theta.numel()),
            "param_count": int(model.theta.numel()),
        }
    return projected_grad_stats(model.theta, model.theta.grad.detach(), min_rate=min_rate, max_rate=max_rate)


def maybe_bump_solver(history, model, *, step, last_adjust_step):
    if STALL_PATIENCE <= 0 or step - last_adjust_step < STALL_PATIENCE or len(history) <= STALL_PATIENCE:
        return False, None, last_adjust_step

    earlier = history[-STALL_PATIENCE - 1]["nll_bits"]
    current = history[-1]["nll_bits"]
    improvement = earlier - current
    required = max(float(STALL_ABS_IMPROVEMENT), float(STALL_REL_IMPROVEMENT) * max(abs(earlier), 1.0))
    if improvement > required:
        return False, None, last_adjust_step

    old_pi = int(model.solver_options.pi_iters)
    old_neumann = int(model.solver_options.neumann_terms)
    pi_cap = max(2, int(PI_ITERS_MAX) - (int(PI_ITERS_MAX) % 2))
    new_pi = min(pi_cap, old_pi + int(PI_ITERS_INCREMENT))
    if new_pi % 2:
        new_pi = min(pi_cap, new_pi + 1)
    new_neumann = min(int(NEUMANN_TERMS_MAX), old_neumann + int(NEUMANN_TERMS_INCREMENT))
    changed = new_pi != old_pi or new_neumann != old_neumann
    if changed:
        model.configure_solver(pi_iters=new_pi, neumann_terms=new_neumann)
        model.clear_warm_starts()
        last_adjust_step = step

    return changed, {
        "solver_changed": changed,
        "old_pi_iters": old_pi,
        "new_pi_iters": int(model.solver_options.pi_iters),
        "old_neumann_terms": old_neumann,
        "new_neumann_terms": int(model.solver_options.neumann_terms),
        "window_improvement_bits": improvement,
        "required_improvement_bits": required,
        "window_start_nll_bits": earlier,
    }, last_adjust_step


def append_row(history, state, row):
    loss_value = row["nll_bits"]
    previous_loss = state.get("previous_loss")
    row["delta_bits"] = None if previous_loss is None else loss_value - previous_loss
    row.update({
        "theta_min": float(model.theta.detach().min().cpu()),
        "theta_max": float(model.theta.detach().max().cpu()),
        "e_max_iter": model.solver_options.e_max_iter,
        "e_tol": model.solver_options.e_tol,
        "pi_iters": model.solver_options.pi_iters,
        "neumann_terms": model.solver_options.neumann_terms,
        "self_loop_solver": model.solver_options.self_loop_solver,
        "bicgstab_max_iter": model.solver_options.bicgstab_max_iter,
        "bicgstab_tol": model.solver_options.bicgstab_tol,
    })
    history.append(row)
    state["previous_loss"] = loss_value


def print_row(row, *, kind="step"):
    delta = row["delta_bits"] if row["delta_bits"] is not None else float("nan")
    projected_grad_linf = row.get("projected_grad_linf", float("nan"))
    active_text = ""
    if "active_params" in row and "free_params" in row:
        active_text = (
            f" active={row['active_params']} free={row['free_params']} "
            f"lower={row.get('active_lower_params', 0)} upper={row.get('active_upper_params', 0)}"
        )
    print(
        f"{kind}={row['step']:04d} nll={row['nll_bits']:.6f} "
        f"obj={row.get('objective_bits', row['nll_bits']):.6f} penalty={row.get('penalty_bits', 0.0):.6f} "
        f"delta={delta:.6f} grad_norm={row['grad_norm']:.3g} "
        f"projected_grad_norm={row['projected_grad_norm']:.3g} "
        f"projected_grad_linf={projected_grad_linf:.3g} "
        f"{active_text} "
        f"pi={row['pi_iters']} self_loop={row['self_loop_solver']}:{row['neumann_terms']} "
        f"optimizer={row['optimizer']} phase={row['optimizer_phase']} "
        f"restarts={row['optimizer_reset_count']} step_s={row['step_s']:.3f}"
    )


def run_first_order_stage(stage, history, state, *, stop_when_solver_ready):
    if stop_when_solver_ready and int(model.solver_options.neumann_terms) >= int(NEUMANN_TERMS_MAX):
        print(f"skipping {stage_label(stage)} because neumann_terms is already {model.solver_options.neumann_terms}")
        return

    optimizer_cls = FIRST_ORDER_OPTIMIZERS[stage]
    optimizer = optimizer_cls([model.theta], lr=LR)
    phase = f"{stage}_until_solver_ready" if stop_when_solver_ready else stage
    clamp_log_rate_(model.theta, min_rate=MIN_RATE, max_rate=MAX_RATE)

    while state["next_step"] <= STEPS:
        step = state["next_step"]
        optimizer.zero_grad(set_to_none=True)
        t0 = time.perf_counter()
        objective, _grad, raw_nll, penalty = evaluate_loss_and_grad(model, min_rate=MIN_RATE, max_rate=MAX_RATE, check_finite=False)

        grad_norm = float(model.theta.grad.detach().norm().cpu())
        projected_grad_stats_row = projected_grad_stats_for_diagnostics(model, min_rate=MIN_RATE, max_rate=MAX_RATE)
        project_rate_gradient_(model.theta, min_rate=MIN_RATE, max_rate=MAX_RATE)
        torch.nn.utils.clip_grad_norm_([model.theta], CLIP_GRAD_NORM)
        projected_grad_norm_after_clip = float(model.theta.grad.detach().norm().cpu())

        optimizer.step()
        clamp_log_rate_(model.theta, min_rate=MIN_RATE, max_rate=MAX_RATE)
        if DEVICE == "cuda":
            torch.cuda.synchronize()

        row = {
            "step": step,
            "nll_bits": float(raw_nll.detach().cpu()),
            "penalty_bits": float(penalty.detach().cpu()),
            "objective_bits": float(objective.detach().cpu()),
            "grad_norm": grad_norm,
            **projected_grad_stats_row,
            "projected_grad_norm_after_clip": projected_grad_norm_after_clip,
            "lr": optimizer.param_groups[0]["lr"],
            "optimizer": stage,
            "optimizer_phase": phase,
            "optimizer_reset_count": state["optimizer_reset_count"],
            "solver_changed": False,
            "step_s": time.perf_counter() - t0,
        }
        append_row(history, state, row)

        solver_changed, solver_info, state["last_solver_adjust_step"] = maybe_bump_solver(
            history, model, step=step, last_adjust_step=state["last_solver_adjust_step"]
        )
        if solver_info:
            row.update(solver_info)
        if solver_changed and RESET_OPTIMIZER_ON_STALL:
            state["optimizer_reset_count"] += 1
            optimizer = optimizer_cls([model.theta], lr=LR)
            row["optimizer_reset_count"] = state["optimizer_reset_count"]
            row["optimizer_restarted"] = True

        solver_ready = int(model.solver_options.neumann_terms) >= int(NEUMANN_TERMS_MAX)
        if stop_when_solver_ready and solver_ready:
            row["phase_transition"] = "solver_ready_for_next_stage"

        if step == 1 or step % PRINT_EVERY == 0 or row.get("solver_changed") or row.get("phase_transition"):
            print_row(row)
        state["next_step"] += 1

        if stop_when_solver_ready and solver_ready:
            print(f"switching {stage_label(stage)} -> next stage because neumann_terms reached {model.solver_options.neumann_terms}")
            break


def run_lbfgsb_stage(history, state):
    try:
        from scipy.optimize import minimize
    except ImportError as exc:
        raise RuntimeError("LBFGSB requires scipy. Install scipy or choose Adam/Adagrad.") from exc

    clamp_log_rate_(model.theta, min_rate=MIN_RATE, max_rate=MAX_RATE)
    x0 = flat_theta_numpy(model)
    maxiter = max(1, int(STEPS - state["next_step"] + 1))

    def objective_and_grad(flat_theta):
        t0 = time.perf_counter()
        step = state["next_step"]
        load_flat_theta_(model, flat_theta, min_rate=MIN_RATE, max_rate=MAX_RATE)
        objective, grad, raw_nll, penalty = evaluate_loss_and_grad(model, min_rate=MIN_RATE, max_rate=MAX_RATE, check_finite=True)
        objective_value = float(objective.detach().cpu())
        raw_nll_value = float(raw_nll.detach().cpu())
        projected_grad_stats_row = projected_grad_stats_for_diagnostics(model, min_rate=MIN_RATE, max_rate=MAX_RATE)
        grad_np = grad.detach().cpu().double().numpy().reshape(-1).copy()

        row = {
            "step": step,
            "nll_bits": raw_nll_value,
            "penalty_bits": float(penalty.detach().cpu()),
            "objective_bits": objective_value,
            "grad_norm": float(grad.detach().norm().cpu()),
            **projected_grad_stats_row,
            "lr": float(LR),
            "optimizer": "lbfgsb",
            "optimizer_phase": "lbfgsb",
            "optimizer_reset_count": state["optimizer_reset_count"],
            "solver_changed": False,
            "step_s": time.perf_counter() - t0,
        }
        append_row(history, state, row)
        if step == 1 or step % PRINT_EVERY == 0:
            print_row(row, kind="eval")
        state["next_step"] += 1
        return objective_value, grad_np

    result = minimize(
        objective_and_grad,
        x0,
        method="L-BFGS-B",
        jac=True,
        bounds=lbfgsb_bounds(model.theta, min_rate=MIN_RATE, max_rate=MAX_RATE),
        options={
            "maxiter": maxiter,
            "maxfun": max(1, int(LBFGSB_MAXFUN_MULTIPLIER) * maxiter),
            "maxls": int(LBFGSB_MAXLS),
            "maxcor": int(LBFGSB_MAXCOR),
            "gtol": float(LBFGSB_GTOL),
            "ftol": float(LBFGSB_FTOL),
        },
    )
    load_flat_theta_(model, result.x, min_rate=MIN_RATE, max_rate=MAX_RATE)
    final_jac = torch.as_tensor(result.jac, dtype=model.theta.dtype, device=model.theta.device).reshape_as(model.theta)
    final_projected_grad_stats = projected_grad_stats(model.theta, final_jac, min_rate=MIN_RATE, max_rate=MAX_RATE)
    if history:
        history[-1].update({
            "lbfgsb_success": bool(result.success),
            "lbfgsb_status": int(result.status),
            "lbfgsb_message": str(result.message),
            "lbfgsb_nit": int(getattr(result, "nit", 0)),
            "lbfgsb_nfev": int(getattr(result, "nfev", 0)),
            "lbfgsb_final_projected_grad_linf": final_projected_grad_stats["projected_grad_linf"],
            "lbfgsb_final_projected_grad_norm": final_projected_grad_stats["projected_grad_norm"],
            "lbfgsb_final_active_params": final_projected_grad_stats["active_params"],
            "lbfgsb_final_free_params": final_projected_grad_stats["free_params"],
            "lbfgsb_final_active_lower_params": final_projected_grad_stats["active_lower_params"],
            "lbfgsb_final_active_upper_params": final_projected_grad_stats["active_upper_params"],
        })
    print(
        f"lbfgsb stopped: status={int(result.status)} success={bool(result.success)} "
        f"nit={int(getattr(result, 'nit', 0))} nfev={int(getattr(result, 'nfev', 0))} "
        f"projected_grad_linf={final_projected_grad_stats['projected_grad_linf']:.3g} "
        f"active={final_projected_grad_stats['active_params']} "
        f"free={final_projected_grad_stats['free_params']} "
        f"lower={final_projected_grad_stats['active_lower_params']} "
        f"upper={final_projected_grad_stats['active_upper_params']} "
        f"message={result.message}"
    )


def optimize_likelihood_chain(name: str):
    chain = parse_optimizer_chain(name)
    print(f"optimizer_chain={chain_label(chain)}")
    history = []
    state = {
        "next_step": 1,
        "previous_loss": None,
        "optimizer_reset_count": 0,
        "last_solver_adjust_step": 0,
    }

    for index, stage in enumerate(chain):
        if state["next_step"] > STEPS:
            break
        if stage in FIRST_ORDER_OPTIMIZERS:
            run_first_order_stage(stage, history, state, stop_when_solver_ready=index < len(chain) - 1)
        elif stage == "lbfgsb":
            run_lbfgsb_stage(history, state)
        else:
            raise ValueError(f"unsupported optimizer stage {stage!r}")
    return history


history = optimize_likelihood_chain(OPTIMIZER_NAME)
history[-1] if history else None


In [ ]:
history

## Plot the trace

In [ ]:
try:
    import matplotlib.pyplot as plt

    steps = [row["step"] for row in history]
    values = [row["nll_bits"] for row in history]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(steps, values, marker=".")
    ax.set_xlabel("step")
    ax.set_ylabel("negative log-likelihood (bits)")
    ax.set_title(f"HOGENOM {MODE} optimization")
    ax.grid(alpha=0.25)
    plt.show()
except ImportError:
    print("matplotlib is not installed; skipping plot")

In [ ]:
history

## Inspect fitted event probabilities

In [ ]:
def event_probabilities(theta: torch.Tensor) -> torch.Tensor:
    theta_cpu = theta.detach().cpu()
    zeros = torch.zeros((*theta_cpu.shape[:-1], 1), dtype=theta_cpu.dtype)
    logits = torch.cat((zeros, theta_cpu), dim=-1)
    return torch.softmax(logits * math.log(2.0), dim=-1)

probs = event_probabilities(model.theta)
labels = ["pT", "pS", "pD", "pL"]

if probs.ndim == 1:
    dict(zip(labels, [float(x) for x in probs]))
else:
    summary = {
        label: {
            "mean": float(probs[..., i].mean()),
            "min": float(probs[..., i].min()),
            "max": float(probs[..., i].max()),
        }
        for i, label in enumerate(labels)
    }
    summary

In [ ]:
summary


## Save trace and final theta

In [ ]:
OUT_DIR = Path(os.environ.get("GPUREC_OPT_OUT_DIR", HOGENOM_ROOT / "output_gpurec_lean_optimization")).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

trace_path = OUT_DIR / f"optimization_trace_{MODE}.json"
theta_path = OUT_DIR / f"theta_{MODE}.pt"

trace_path.write_text(json.dumps(history, indent=2) + "\n")
torch.save(model.theta.detach().cpu(), theta_path)

trace_path, theta_path